# Model Prediction Task I


## Imports

Import the required libraries.


In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

import os
from time import perf_counter

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor


## Initial data exploration


### Preview random source CSV records

Display each CSV’s shape and a random sample of records with all columns visible.

<b>Observations of Inconsistency</b>:
- 2015-2016 and 2017 csvs have an additional column `remaining_lease`.
    - 2015-2016 express its values in `year` format, 2017 expresses in `year` and `month` format. 
    - This column will be retained during concatenation and decided subsequently whether they will be used in evaluation.
- At first glance, `flat_model` values are inconsistent, where 1990-1999 takes on small casing while the rest are in upper casing.

In [2]:
data_dir = Path("data")
csv_files = sorted(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV files found in data/. Run from the project root.")

# Sample 5 records from each CSV
for file in csv_files:
    print(f"\n{file.name}")
    source_df = pd.read_csv(
        file,
        dtype={"block": "string", "remaining_lease": "string"},
    )
    print(f"Shape: {source_df.shape[0]:,} rows × {source_df.shape[1]} columns")
    preview = source_df.sample(n=min(5, len(source_df)), random_state=42)
    with pd.option_context("display.max_columns", None):
        display(preview)
    del source_df



resale-flat-prices-based-on-approval-date-1990-1999.csv
Shape: 287,200 rows × 10 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
212281,1998-09,SERANGOON,5 ROOM,242,SERANGOON AVE 3,07 TO 09,121.0,IMPROVED,1986,388000
23470,1991-11,CLEMENTI,3 ROOM,335,CLEMENTI AVE 2,04 TO 06,67.0,NEW GENERATION,1978,47000
214059,1998-10,ANG MO KIO,5 ROOM,111,ANG MO KIO AVE 4,07 TO 09,117.0,IMPROVED,1978,365000
116387,1996-02,MARINE PARADE,3 ROOM,32,MARINE CRES,13 TO 15,76.0,IMPROVED,1975,230000
195038,1998-05,SERANGOON,5 ROOM,213,SERANGOON AVE 4,07 TO 09,121.0,IMPROVED,1985,385000



resale-flat-prices-based-on-approval-date-2000-feb-2012.csv
Shape: 369,651 rows × 10 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
189924,2005-09,YISHUN,4 ROOM,879,YISHUN ST 81,04 TO 06,104.0,Model A,1987,182000.0
179578,2005-05,TOA PAYOH,4 ROOM,213,LOR 8 TOA PAYOH,04 TO 06,82.0,Improved,1975,210000.0
131026,2003-10,PASIR RIS,EXECUTIVE,714,PASIR RIS ST 72,07 TO 09,148.0,Apartment,1996,446000.0
221311,2006-11,HOUGANG,EXECUTIVE,374,HOUGANG ST 31,13 TO 15,141.0,Apartment,2000,425000.0
342018,2010-11,YISHUN,3 ROOM,394,YISHUN AVE 6,04 TO 06,73.0,Model A,1989,275000.0



resale-flat-prices-based-on-registration-date-from-jan-2015-to-dec-2016.csv
Shape: 37,153 rows × 11 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
70,2015-01,BEDOK,3 ROOM,542,BEDOK NTH ST 3,10 TO 12,67.0,New Generation,1985,69,285000.0
9632,2015-07,QUEENSTOWN,4 ROOM,28B,DOVER CRES,22 TO 24,88.0,Model A,2012,95,738888.0
33141,2016-10,GEYLANG,5 ROOM,39,JLN TIGA,07 TO 09,111.0,Improved,2005,87,840000.0
12346,2015-09,JURONG WEST,4 ROOM,662B,JURONG WEST ST 64,07 TO 09,91.0,Model A,2001,84,435000.0
13260,2015-10,BEDOK,3 ROOM,107,BEDOK NTH RD,10 TO 12,67.0,New Generation,1977,60,304888.0



resale-flat-prices-based-on-registration-date-from-jan-2017-onwards.csv
Shape: 80,374 rows × 11 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
78124,2020-09,BEDOK,5 ROOM,19,BEDOK STH RD,22 TO 24,114.0,Standard,1976,54 years 05 months,630000.0
20052,2017-12,SENGKANG,4 ROOM,106,RIVERVALE WALK,01 TO 03,100.0,Model A,1999,80 years 02 months,350000.0
71383,2020-06,GEYLANG,4 ROOM,324,UBI AVE 1,10 TO 12,84.0,Simplified,1985,64 years 02 months,382000.0
2188,2017-02,YISHUN,3 ROOM,237,YISHUN RING RD,01 TO 03,67.0,New Generation,1985,67 years 04 months,260000.0
18795,2017-11,WOODLANDS,5 ROOM,523,WOODLANDS DR 14,07 TO 09,125.0,Premium Apartment,2000,81 years 07 months,443000.0



resale-flat-prices-based-on-registration-date-from-mar-2012-to-dec-2014.csv
Shape: 52,203 rows × 10 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
27423,2013-06,BUKIT MERAH,4 ROOM,8B,BOON TIONG RD,07 TO 09,100.0,Model A,2005,790000.0
50262,2014-11,PASIR RIS,4 ROOM,452,PASIR RIS DR 6,10 TO 12,104.0,Model A,1989,430000.0
51017,2014-12,BEDOK,4 ROOM,703,BEDOK RESERVOIR RD,01 TO 03,93.0,New Generation,1980,391000.0
9273,2012-07,BUKIT MERAH,4 ROOM,112,BT PURMEI RD,10 TO 12,103.0,Model A,1985,618000.0
49936,2014-11,CLEMENTI,3 ROOM,510,WEST COAST DR,04 TO 06,67.0,New Generation,1978,318000.0


### Combine and sort transactions

Combine all CSVs, retain all columns, and sort by month from oldest to newest.


In [3]:
frames = []
for file in csv_files:
    frame = pd.read_csv(
        file,
        dtype={"block": "string", "remaining_lease": "string"},
    )
    frames.append(frame)

# Align columns by name and retain every source record.
df = pd.concat(frames, ignore_index=True, sort=False)
del frames, frame

df["month"] = pd.to_datetime(df["month"], format="%Y-%m", errors="raise").dt.to_period("M")
df = df.sort_values("month", kind="stable").reset_index(drop=True)

print(f"Combined shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Month range: {df['month'].min()} to {df['month'].max()}")
with pd.option_context("display.max_columns", None):
    display(df)

Combined shape: 826,581 rows × 11 columns
Month range: 1990-01 to 2020-09


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
826576,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,Improved,1987,440000.0,66 years 03 months
826577,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,Improved,1987,458000.0,65 years 06 months
826578,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,Improved,1987,490000.0,66 years 04 months
826579,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,Maisonette,1987,558000.0,66 years 03 months


### Save initial combined data

Save the combined dataset before cleaning as `resale_flat_prices_initial.parquet`.


In [4]:
combined_path = Path("data/model_prediction/interim/resale_flat_prices_initial.parquet")
combined_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(combined_path, index=False)

## Data Cleaning

### Load initial data

Read the initial Parquet dataset with its saved column types.


In [5]:
df_clean = pd.read_parquet("data/model_prediction/interim/resale_flat_prices_initial.parquet")
df_clean

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
826576,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,Improved,1987,440000.0,66 years 03 months
826577,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,Improved,1987,458000.0,65 years 06 months
826578,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,Improved,1987,490000.0,66 years 04 months
826579,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,Maisonette,1987,558000.0,66 years 03 months


### Inspect column types

Display the data type of each column in the combined dataset.


In [6]:
dtype_summary = df_clean.dtypes.rename_axis("column").reset_index(name="data_type")
dtype_summary

,column,data_type
0,month,period[M]
1,town,str
2,flat_type,str
3,block,string
4,street_name,str
5,storey_range,str
6,floor_area_sqm,float64
7,flat_model,str
8,lease_commence_date,int64
9,resale_price,float64


### Inspect value counts

Show value counts for each column, including missing values.


In [7]:
for column in df_clean.columns:
    print(f"\n{column}")
    display(df_clean[column].value_counts(dropna=False).reset_index(name="count"))


month


,month,count
0,1999-03,6468
1,1999-06,5862
2,1998-10,5709
3,1999-04,5698
4,1999-05,5671
...,...,...
364,1991-02,816
365,1990-10,810
366,1990-05,778
367,2020-04,424



town


,town,count
0,TAMPINES,73101
1,YISHUN,63309
2,BEDOK,61400
3,JURONG WEST,60708
4,WOODLANDS,58735
5,ANG MO KIO,48237
6,HOUGANG,45476
7,BUKIT BATOK,40444
8,CHOA CHU KANG,33388
9,BUKIT MERAH,30518



flat_type


,flat_type,count
0,4 ROOM,309314
1,3 ROOM,272580
2,5 ROOM,170408
3,EXECUTIVE,62641
4,2 ROOM,9863
5,1 ROOM,1273
6,MULTI GENERATION,279
7,MULTI-GENERATION,223


... (12 outputs left out)


resale_price


,resale_price,count
0,300000.0,6413
1,280000.0,6253
2,250000.0,6151
3,350000.0,5996
4,320000.0,5970
...,...,...
8476,1158000.0,1
8477,326168.0,1
8478,822888.0,1
8479,671300.0,1



remaining_lease


,remaining_lease,count
0,<NA>,709054
1,68,2328
2,67,1857
3,69,1635
4,71,1547
...,...,...
669,93 years 0 months,1
670,97 years 01 month,1
671,97 years 09 months,1
672,45 years 04 months,1


### Check missing values

Display the missing-value count for each column.


In [8]:
df_clean.isna().sum().rename_axis("column").reset_index(name="missing_count")

,column,missing_count
0,month,0
1,town,0
2,flat_type,0
3,block,0
4,street_name,0
5,storey_range,0
6,floor_area_sqm,0
7,flat_model,0
8,lease_commence_date,0
9,resale_price,0


### Check and remove duplicated rows

Count duplicates beyond the first occurrence and preview 10.


In [9]:
duplicates = df_clean.duplicated()
print(f"Duplicate rows: {duplicates.sum():,}")
df_clean.loc[duplicates].head(10)

Duplicate rows: 1,821


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
673,1990-01,GEYLANG,3 ROOM,47,CIRCUIT RD,01 TO 03,56.0,STANDARD,1969,18000.0,<NA>
726,1990-01,HOUGANG,3 ROOM,308,HOUGANG AVE 5,10 TO 12,67.0,NEW GENERATION,1983,47000.0,<NA>
843,1990-01,JURONG WEST,3 ROOM,145,HU CHING RD,04 TO 06,64.0,IMPROVED,1976,23400.0,<NA>
854,1990-01,JURONG WEST,3 ROOM,172,BOON LAY DR,10 TO 12,70.0,IMPROVED,1974,32900.0,<NA>
862,1990-01,JURONG WEST,3 ROOM,185,BOON LAY AVE,22 TO 24,59.0,IMPROVED,1975,22500.0,<NA>
867,1990-01,JURONG WEST,3 ROOM,186,BOON LAY AVE,07 TO 09,59.0,IMPROVED,1975,31500.0,<NA>
877,1990-01,JURONG WEST,3 ROOM,187,BOON LAY AVE,19 TO 21,68.0,IMPROVED,1978,26000.0,<NA>
968,1990-01,JURONG WEST,EXECUTIVE,407,JURONG WEST ST 42,10 TO 12,150.0,MAISONETTE,1985,160000.0,<NA>
1097,1990-01,QUEENSTOWN,3 ROOM,2,GHIM MOH RD,10 TO 12,65.0,IMPROVED,1976,40000.0,<NA>
1098,1990-01,QUEENSTOWN,3 ROOM,2,GHIM MOH RD,04 TO 06,65.0,IMPROVED,1976,36000.0,<NA>


#### Remove duplicate rows

Keep the first occurrence of each row and show the original, removed and remaining counts.


In [10]:
original_count = len(df_clean)
df_clean.drop_duplicates(ignore_index=True, inplace=True)

print(f"Original rows: {original_count:,}")
print(f"Removed rows: {original_count - len(df_clean):,}")
print(f"Latest rows: {len(df_clean):,}")

duplicates = df_clean.duplicated()
print(f"Duplicate rows: {duplicates.sum():,}")
df_clean.loc[duplicates].head(10)

/var/folders/8l/_7g518rx3js3_kngmmslndl40000gn/T/ipykernel_23237/1901161330.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.drop_duplicates(ignore_index=True, inplace=True)


Original rows: 826,581
Removed rows: 1,821
Latest rows: 824,760
Duplicate rows: 0


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease


### Standardise `flat_type`

Replace `MULTI GENERATION` with `MULTI-GENERATION`, leaving other flat types unchanged.


In [11]:
df_clean["flat_type"] = df_clean["flat_type"].replace({"MULTI GENERATION": "MULTI-GENERATION"})
df_clean["flat_type"].value_counts()

flat_type
4 ROOM              308753
3 ROOM              271685
5 ROOM              170177
EXECUTIVE            62584
2 ROOM                9817
1 ROOM                1242
MULTI-GENERATION       502
Name: count, dtype: int64

### Standardise `flat_model` letter casing

Trim spaces, uppercase flat model labels, and show the DataFrame.


In [12]:
df_clean["flat_model"] = df_clean["flat_model"].astype("string").str.strip().str.upper()
df_clean["flat_model"].value_counts()

flat_model
MODEL A                   227961
IMPROVED                  216775
NEW GENERATION            177121
SIMPLIFIED                 53854
STANDARD                   39761
PREMIUM APARTMENT          34947
APARTMENT                  31979
MAISONETTE                 26881
MODEL A2                    9090
MODEL A-MAISONETTE          1888
DBSS                        1607
ADJOINED FLAT               1084
TERRACE                      642
MULTI GENERATION             502
TYPE S1                      272
TYPE S2                      129
IMPROVED-MAISONETTE          114
PREMIUM MAISONETTE            82
2-ROOM                        40
PREMIUM APARTMENT LOFT        31
Name: count, dtype: int64[pyarrow]

## Feature Engineering

### Standardise `remaining_lease` to months

Convert lease to months and show the month and both lease columns for converted rows in `remaining_lease_months`. There are no current plans to use this feature.


In [13]:
df = df_clean

lease = df["remaining_lease"].astype("string").str.strip().str.lower()
parts = lease.str.extract(r"^(\d+)(?:\s+years?(?:\s+(\d+)\s+months?)?)?$")
assert (lease.isna() | parts[0].notna()).all(), "Unexpected remaining lease format"

df["remaining_lease_months"] = (
    parts[0].astype("Int64") * 12 + parts[1].astype("Int64").fillna(0)
)

df.loc[df["remaining_lease_months"].notna()]

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,remaining_lease_months
707467,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,07 TO 09,60.0,IMPROVED,1986,255000.0,70,840
707468,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,01 TO 03,68.0,NEW GENERATION,1981,275000.0,65,780
707469,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,01 TO 03,69.0,NEW GENERATION,1980,285000.0,64,768
707470,2015-01,ANG MO KIO,3 ROOM,446,ANG MO KIO AVE 10,01 TO 03,68.0,NEW GENERATION,1979,290000.0,63,756
707471,2015-01,ANG MO KIO,3 ROOM,557,ANG MO KIO AVE 10,07 TO 09,68.0,NEW GENERATION,1980,290000.0,64,768
...,...,...,...,...,...,...,...,...,...,...,...,...
824755,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,IMPROVED,1987,440000.0,66 years 03 months,795
824756,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,IMPROVED,1987,458000.0,65 years 06 months,786
824757,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,IMPROVED,1987,490000.0,66 years 04 months,796
824758,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,MAISONETTE,1987,558000.0,66 years 03 months,795


### Extract transaction year and month

Create year and month-of-year features while retaining the original date.


In [14]:
df["transaction_year"] = df["month"].dt.year
df["transaction_month"] = df["month"].dt.month
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,remaining_lease_months,transaction_year,transaction_month
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>,<NA>,1990,1
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>,<NA>,1990,1
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>,<NA>,1990,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824755,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,IMPROVED,1987,440000.0,66 years 03 months,795,2020,9
824756,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,IMPROVED,1987,458000.0,65 years 06 months,786,2020,9
824757,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,IMPROVED,1987,490000.0,66 years 04 months,796,2020,9
824758,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,MAISONETTE,1987,558000.0,66 years 03 months,795,2020,9


### Extract storey features

Extract floor bounds, midpoint and width (max - min), retaining the original range.


In [15]:
df[["storey_min", "storey_max"]] = (
    df["storey_range"].str.split(" TO ", expand=True).astype(int)
)
df["storey_midpoint"] = ((df["storey_min"] + df["storey_max"]) / 2).astype(int)

df["storey_width"] = df["storey_max"] - df["storey_min"]
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,remaining_lease_months,transaction_year,transaction_month,storey_min,storey_max,storey_midpoint,storey_width
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>,<NA>,1990,1,10,12,11,2
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,4,6,5,2
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>,<NA>,1990,1,10,12,11,2
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,7,9,8,2
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>,<NA>,1990,1,4,6,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824755,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,IMPROVED,1987,440000.0,66 years 03 months,795,2020,9,7,9,8,2
824756,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,IMPROVED,1987,458000.0,65 years 06 months,786,2020,9,7,9,8,2
824757,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,IMPROVED,1987,490000.0,66 years 04 months,796,2020,9,4,6,5,2
824758,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,MAISONETTE,1987,558000.0,66 years 03 months,795,2020,9,4,6,5,2


### Calculate flat age

Estimate age since lease commencement as transaction year minus lease commencement year.


In [16]:
df["flat_age_years"] = df["transaction_year"] - df["lease_commence_date"]
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,remaining_lease_months,transaction_year,transaction_month,storey_min,storey_max,storey_midpoint,storey_width,flat_age_years
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>,<NA>,1990,1,10,12,11,2,13
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,4,6,5,2,13
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>,<NA>,1990,1,10,12,11,2,13
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,7,9,8,2,13
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>,<NA>,1990,1,4,6,5,2,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824755,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,IMPROVED,1987,440000.0,66 years 03 months,795,2020,9,7,9,8,2,33
824756,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,IMPROVED,1987,458000.0,65 years 06 months,786,2020,9,7,9,8,2,33
824757,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,IMPROVED,1987,490000.0,66 years 04 months,796,2020,9,4,6,5,2,33
824758,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,MAISONETTE,1987,558000.0,66 years 03 months,795,2020,9,4,6,5,2,33


### Create block address

Combine block and street name into a location identifier, not a unique flat or transaction ID.


In [17]:
df["block_address"] = (
    df["block"].astype("string").str.strip()
    + " "
    + df["street_name"].astype("string").str.strip().str.upper()
)
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,remaining_lease_months,transaction_year,transaction_month,storey_min,storey_max,storey_midpoint,storey_width,flat_age_years,block_address
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>,<NA>,1990,1,10,12,11,2,13,309 ANG MO KIO AVE 1
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,4,6,5,2,13,309 ANG MO KIO AVE 1
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>,<NA>,1990,1,10,12,11,2,13,309 ANG MO KIO AVE 1
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,7,9,8,2,13,309 ANG MO KIO AVE 1
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>,<NA>,1990,1,4,6,5,2,14,216 ANG MO KIO AVE 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824755,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,IMPROVED,1987,440000.0,66 years 03 months,795,2020,9,7,9,8,2,33,716 YISHUN ST 71
824756,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,IMPROVED,1987,458000.0,65 years 06 months,786,2020,9,7,9,8,2,33,760 YISHUN ST 72
824757,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,IMPROVED,1987,490000.0,66 years 04 months,796,2020,9,4,6,5,2,33,835 YISHUN ST 81
824758,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,MAISONETTE,1987,558000.0,66 years 03 months,795,2020,9,4,6,5,2,33,791 YISHUN AVE 2


### Save final dataset

Save the cleaned, feature-engineered DataFrame as `resale_flat_prices_final.parquet`.


In [18]:
final_path = Path("data/model_prediction/processed/resale_flat_prices_final.parquet")
final_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(final_path, index=False)

### Summary statistics

Summarise numeric columns with counts, averages, spread, quartiles and minimum/maximum values.


In [19]:
df.describe()

,floor_area_sqm,lease_commence_date,resale_price,remaining_lease_months,transaction_year,transaction_month,storey_min,storey_max,storey_midpoint,storey_width,flat_age_years
count,824760.000000,824760.000000,8.247600e+05,117293.0,824760.000000,824760.000000,824760.000000,824760.000000,824760.000000,824760.000000,824760.000000
mean,95.583872,1987.153285,2.936472e+05,893.362946,2004.237204,6.565789,6.535369,8.551936,7.543653,2.016567,17.083919
std,26.054676,9.448888,1.487293e+05,146.851205,7.924290,3.391979,4.623421,4.623947,4.622796,0.181273,9.808938
min,28.000000,1966.000000,5.000000e+03,544.0,1990.000000,1.000000,1.000000,3.000000,2.000000,2.000000,-2.000000
25%,73.000000,1980.000000,1.810000e+05,781.0,1998.000000,4.000000,4.000000,6.000000,5.000000,2.000000,9.000000
50%,93.000000,1986.000000,2.750000e+05,887.0,2003.000000,7.000000,7.000000,9.000000,8.000000,2.000000,15.000000
75%,114.000000,1994.000000,3.830000e+05,1002.0,2010.000000,9.000000,10.000000,12.000000,11.000000,2.000000,24.000000
max,307.000000,2019.000000,1.258000e+06,1173.0,2020.000000,12.000000,49.000000,51.000000,50.000000,4.000000,54.000000


## Exploratory data analysis

### Load final dataset

Load the cleaned, feature-engineered Parquet dataset for EDA.


In [20]:
df = pd.read_parquet("data/model_prediction/processed/resale_flat_prices_final.parquet")
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,remaining_lease_months,transaction_year,transaction_month,storey_min,storey_max,storey_midpoint,storey_width,flat_age_years,block_address
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,<NA>,<NA>,1990,1,10,12,11,2,13,309 ANG MO KIO AVE 1
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,4,6,5,2,13,309 ANG MO KIO AVE 1
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,<NA>,<NA>,1990,1,10,12,11,2,13,309 ANG MO KIO AVE 1
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,<NA>,<NA>,1990,1,7,9,8,2,13,309 ANG MO KIO AVE 1
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,<NA>,<NA>,1990,1,4,6,5,2,14,216 ANG MO KIO AVE 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824755,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,IMPROVED,1987,440000.0,66 years 03 months,795,2020,9,7,9,8,2,33,716 YISHUN ST 71
824756,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,IMPROVED,1987,458000.0,65 years 06 months,786,2020,9,7,9,8,2,33,760 YISHUN ST 72
824757,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,IMPROVED,1987,490000.0,66 years 04 months,796,2020,9,4,6,5,2,33,835 YISHUN ST 81
824758,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,MAISONETTE,1987,558000.0,66 years 03 months,795,2020,9,4,6,5,2,33,791 YISHUN AVE 2


### Identify unexpected values/outliers

Visualise feature distributions to identify unexpected values and potential outliers.

In [21]:
# Group categorical charts first, quantitative charts next, and time last.
fig = make_subplots(
    rows=5, cols=2, horizontal_spacing=0.22, vertical_spacing=0.045,
    specs=[[{}, {}], [{"colspan": 2}, None], [{}, {}], [{}, {}], [{"colspan": 2}, None]],
    row_heights=[0.27, 0.10, 0.20, 0.20, 0.19],
    subplot_titles=["Town distribution", "Flat model distribution", "Flat type distribution",
                    "Floor area distribution", "Flat age distribution", "Resale price distribution",
                    "Resale price outliers", "Transactions by month"],
)

# Compare transaction counts across the three categorical features.
for column, row, col in [("town", 1, 1), ("flat_model", 1, 2), ("flat_type", 2, 1)]:
    counts = df[column].value_counts().sort_values()
    fig.add_trace(go.Bar(x=counts.values, y=counts.index, orientation="h"), row=row, col=col)
    fig.update_yaxes(automargin=True, tickfont_size=10, row=row, col=col)

# Group floor areas into 30 bins, calculated in Python.
area_counts = pd.cut(df["floor_area_sqm"].dropna(), bins=30).value_counts(sort=False)
fig.add_trace(go.Bar(
    x=[interval.mid for interval in area_counts.index], y=area_counts.values,
    width=[interval.length for interval in area_counts.index],
    customdata=area_counts.index.astype(str),
    hovertemplate="Range: %{customdata}<br>Transactions: %{y:,}<extra></extra>",
), row=3, col=1)

# Use one-year bins centred on each whole-number flat age.
age_counts = df["flat_age_years"].value_counts().sort_index()
fig.add_trace(go.Bar(
    x=age_counts.index, y=age_counts.values, width=1,
    hovertemplate="Age: %{x:.0f} years<br>Transactions: %{y:,}<extra></extra>",
), row=3, col=2)

# Use SGD 25,000 price bins covering the entire observed price range.
prices = df["resale_price"].dropna()
price_bin_width = 25_000
price_start = int(prices.min() // price_bin_width) * price_bin_width
price_stop = (int(prices.max() // price_bin_width) + 1) * price_bin_width
price_edges = range(price_start, price_stop + price_bin_width, price_bin_width)
price_counts = pd.cut(prices, bins=price_edges, right=False).value_counts(sort=False)
fig.add_trace(go.Bar(
    x=[interval.mid for interval in price_counts.index], y=price_counts.values,
    width=price_bin_width,
    customdata=[[interval.left, interval.right] for interval in price_counts.index],
    hovertemplate="Price: SGD %{customdata[0]:,.0f} to under %{customdata[1]:,.0f}<br>Transactions: %{y:,}<extra></extra>",
), row=4, col=1)

# Flag potential resale-price outliers using the boxplot's 1.5-IQR whiskers.
fig.add_trace(go.Box(
    x=prices, orientation="h", boxpoints="outliers", quartilemethod="linear",
    marker={"size": 4, "opacity": 0.5},
    line_color="#1f77b4", fillcolor="rgba(31,119,180,0.25)",
), row=4, col=2)

# Track monthly transaction counts in a separate full-width time panel.
monthly_counts = df["month"].value_counts().sort_index()
fig.add_trace(go.Scatter(x=monthly_counts.index.to_timestamp(), y=monthly_counts.values,
                         mode="lines"), row=5, col=1)

# Label each panel and format numeric axes.
for row, col, xlabel, ylabel in [
    (1, 1, "Transactions", "Town"),
    (1, 2, "Transactions", "Flat model"),
    (2, 1, "Transactions", "Flat type"),
    (3, 1, "Floor area (sqm)", "Transactions"),
    (3, 2, "Approximate age since lease commencement (years)", "Transactions"),
    (4, 1, "Resale price (SGD)", "Transactions"),
    (4, 2, "Resale price (SGD)", ""),
    (5, 1, "Transaction month", "Transactions"),
]:
    fig.update_xaxes(title_text=xlabel, row=row, col=col)
    fig.update_yaxes(title_text=ylabel, row=row, col=col)

fig.update_yaxes(showticklabels=False, row=4, col=2)
fig.update_xaxes(tickformat=",.0f", row=4, col=1)
fig.update_xaxes(tickformat=",.0f", row=4, col=2)
fig.update_xaxes(tickformat="d", dtick=5, row=3, col=2)

# Keep charts blue, hide trace names, and resize to the notebook width.
fig.update_traces(name="", hoverlabel_namelength=0, marker_color="#1f77b4")
fig.update_traces(line_color="#1f77b4", selector={"type": "scatter"})
fig.update_layout(height=2100, autosize=True, showlegend=False, template="plotly_white")
fig.show(renderer="plotly_mimetype", config={"responsive": True})

### Correlation heatmap

Show Pearson correlations for all numeric columns; text columns are excluded and missing values are handled pairwise.

**Observations:**
- **Resale price and floor area (0.62):** Larger flats generally sell for higher prices.
- **Resale price and transaction year (0.60):** Flats sold in later years tend to have higher prices, although changes in the mix of flats sold may contribute to this trend.
- **Resale price and lease commencement year (0.50):** Flats with more recent lease start years tend to sell for higher prices, but size, location and the year of sale may also explain this pattern.
- **Resale price and remaining lease months (0.34):** Longer remaining leases have a weak positive relationship with price. This uses only records from 2015 onwards, where remaining lease is available.


In [22]:
# Compute Pearson correlations using available numeric pairs.
correlations = df.select_dtypes(include="number").corr()

# Show signed correlations on a fixed -1 to 1 scale.
correlation_fig = go.Figure(go.Heatmap(
    z=correlations.values,
    x=correlations.columns,
    y=correlations.index,
    zmin=-1, zmax=1, colorscale="Blues",
    texttemplate="%{z:.2f}",
    hovertemplate="%{y} vs %{x}<br>Correlation: %{z:.3f}<extra></extra>",
    colorbar={"title": "Pearson r"},
))
correlation_fig.update_layout(
    title="Numeric feature correlations", height=800, autosize=True,
    template="plotly_white", margin={"l": 180, "b": 180},
)
correlation_fig.update_xaxes(tickangle=-45)
correlation_fig.update_yaxes(autorange="reversed")
correlation_fig.show(renderer="plotly_mimetype", config={"responsive": True})


### Price versus floor area

Show all recorded years with a median line and middle-50% band in 10-sqm groups. Summaries use all valid records, with at least 30 per band; background points are sampled (up to 10,000, seed 42).

**Observation:** Prices generally increase with floor area, but flats of similar sizes can sell for very different prices. Location, remaining lease and sale year may also play a part in explaining these price differences.


In [23]:
# Use all recorded years and calculate 10-sqm summaries from all valid records.
area_price_data = df[["floor_area_sqm", "resale_price"]].dropna()
area_price_data = area_price_data.assign(area_bin=(area_price_data["floor_area_sqm"] // 10 * 10).astype(int))
area_summary = area_price_data.groupby("area_bin")["resale_price"].agg(
    median="median", q1=lambda prices: prices.quantile(0.25),
    q3=lambda prices: prices.quantile(0.75), transactions="count",
)
area_summary = area_summary.reindex(range(int(area_summary.index.min()), int(area_summary.index.max()) + 10, 10))

# Leave sparse bands as gaps instead of presenting unstable summaries.
area_summary[["median", "q1", "q3"]] = area_summary[["median", "q1", "q3"]].where(area_summary["transactions"] >= 30, axis=0)
area_summary = area_summary.rename_axis("area_bin").reset_index()
area_summary["bin_end"] = area_summary["area_bin"] + 10
area_centres = area_summary["area_bin"] + 5
scatter_data = area_price_data.sample(n=min(10_000, len(area_price_data)), random_state=42)

# Shade the middle 50% of prices, not a confidence interval.
scatter_fig = go.Figure(go.Scatter(
    x=area_centres, y=area_summary["q1"], mode="lines", line={"width": 0},
    showlegend=False, hoverinfo="skip", connectgaps=False,
))
scatter_fig.add_trace(go.Scatter(
    x=area_centres, y=area_summary["q3"], mode="lines", line={"width": 0},
    fill="tonexty", fillcolor="rgba(31,119,180,0.2)", name="Middle 50% of prices",
    hoverinfo="skip", connectgaps=False,
))

# Sample only the background points to keep the chart readable.
scatter_fig.add_trace(go.Scatter(
    x=scatter_data["floor_area_sqm"], y=scatter_data["resale_price"], mode="markers",
    name="Sampled transactions", marker={"size": 4, "color": "#1f77b4", "opacity": 0.2},
    hovertemplate="Area: %{x} sqm<br>Price: SGD %{y:,.0f}<extra></extra>",
))

# Overlay median prices at each band's centre and show full-data counts on hover.
scatter_fig.add_trace(go.Scatter(
    x=area_centres, y=area_summary["median"], mode="lines+markers", name="Median price",
    line={"color": "#125183", "width": 3}, marker={"size": 6}, connectgaps=False,
    customdata=area_summary[["area_bin", "bin_end", "transactions", "q1", "q3"]],
    hovertemplate="Area: %{customdata[0]:.0f} to under %{customdata[1]:.0f} sqm<br>Median: SGD %{y:,.0f}<br>Middle 50%: SGD %{customdata[3]:,.0f}–%{customdata[4]:,.0f}<br>Transactions: %{customdata[2]:,.0f}<extra></extra>",
))
scatter_fig.update_layout(
    title="Price vs floor area — all years (10-sqm bands)",
    xaxis_title="Floor area (sqm)", yaxis_title="Resale price (SGD)",
    height=650, autosize=True, template="plotly_white", legend={"orientation": "h", "y": 1.08},
)
scatter_fig.update_yaxes(tickformat=",.0f")
scatter_fig.show(renderer="plotly_mimetype", config={"responsive": True})


### Price trends by flat type

Plot annual median prices by flat type; gaps mark groups with fewer than 30 records. 2020 covers January–September.

**Observation:** Most flat types follow a similar pattern of price rises and falls, suggesting broader market conditions may play a part alongside flat characteristics. Median prices are generally higher for larger flat types, but growth has not been continuous, with declines after the late-1990s and early-2010s peaks.


In [24]:
# Calculate annual medians and sample sizes for each flat type.
annual_type_prices = df.groupby(["transaction_year", "flat_type"]).agg(
    median_price=("resale_price", "median"), transactions=("resale_price", "count")
).reset_index()
trend_years = list(range(int(df["transaction_year"].min()), int(df["transaction_year"].max()) + 1))
type_palette = ["#08306b", "#08519c", "#2171b5", "#4292c6", "#6baed6", "#9ecae1", "#c6dbef"]
price_type_fig = go.Figure()

# Keep low-count estimates as gaps rather than joining across them.
for i, (flat_type, group) in enumerate(annual_type_prices.groupby("flat_type")):
    group = group.set_index("transaction_year").reindex(trend_years)
    price_type_fig.add_trace(go.Scatter(
        x=trend_years, y=group["median_price"].where(group["transactions"] >= 30),
        mode="lines+markers", name=flat_type, connectgaps=False,
        line={"color": type_palette[i], "width": 2}, marker={"size": 4},
        customdata=group[["transactions"]],
        hovertemplate="Year: %{x}<br>Median: SGD %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>",
    ))
price_type_fig.update_layout(
    title="Median resale price by flat type", xaxis_title="Recorded transaction year",
    yaxis_title="Median resale price (SGD)", template="plotly_white", height=600, autosize=True,
    legend_title_text="Flat type",
)
price_type_fig.update_xaxes(tickvals=trend_years, ticktext=["2020 (Jan–Sep)" if y == 2020 else str(y) for y in trend_years], tickangle=-45)
price_type_fig.update_yaxes(tickformat=",.0f")
price_type_fig.show(renderer="plotly_mimetype", config={"responsive": True})


### Price per square metre trends by flat type

Plot annual median price per sqm by flat type, showing groups with at least 30 records. Price per sqm is calculated locally for EDA only, not added to the modelling DataFrame.

**Observation:** Smaller flats do not always cost less per sqm. From around 2008 onwards, 2-room and 3-room flats generally had higher median prices per sqm than larger flat types. This suggests that a lower total purchase price does not necessarily mean a lower cost for the space, although location and lease differences may also play a part.


In [25]:
# Calculate price per sqm only in a separate EDA table.
price_sqm_data = df.loc[df["floor_area_sqm"] > 0, [
    "transaction_year", "flat_type", "resale_price", "floor_area_sqm"
]].assign(price_per_sqm=lambda data: (data["resale_price"] / data["floor_area_sqm"]).round(2))

# Summarise the analysis measure without adding it to df.
annual_sqm_prices = price_sqm_data.groupby(["transaction_year", "flat_type"]).agg(
    median_price_per_sqm=("price_per_sqm", "median"), transactions=("price_per_sqm", "count")
).reset_index()
sqm_years = list(range(int(df["transaction_year"].min()), int(df["transaction_year"].max()) + 1))
sqm_palette = ["#08306b", "#08519c", "#2171b5", "#4292c6", "#6baed6", "#9ecae1", "#c6dbef"]
sqm_trend_fig = go.Figure()

# Compare trends within flat types, leaving sparse groups as gaps.
for i, (flat_type, group) in enumerate(annual_sqm_prices.groupby("flat_type")):
    group = group.set_index("transaction_year").reindex(sqm_years)
    sqm_trend_fig.add_trace(go.Scatter(
        x=sqm_years, y=group["median_price_per_sqm"].where(group["transactions"] >= 30),
        mode="lines+markers", name=flat_type, connectgaps=False,
        line={"color": sqm_palette[i], "width": 2}, marker={"size": 4},
        customdata=group[["transactions"]],
        hovertemplate="Year: %{x}<br>Median: SGD %{y:,.0f}/sqm<br>Transactions: %{customdata[0]:,.0f}<extra></extra>",
    ))
sqm_trend_fig.update_layout(
    title="Median resale price per sqm by flat type", xaxis_title="Recorded transaction year",
    yaxis_title="Median price per sqm (SGD)", template="plotly_white", height=600, autosize=True,
    legend_title_text="Flat type",
)
sqm_trend_fig.update_xaxes(tickvals=sqm_years, ticktext=["2020 (Jan–Sep)" if y == 2020 else str(y) for y in sqm_years], tickangle=-45)
sqm_trend_fig.update_yaxes(tickformat=",.0f")
sqm_trend_fig.show(renderer="plotly_mimetype", config={"responsive": True})


### Town prices by room flat type

Compare 3-room, 4-room and 5-room median prices across towns using January 2016–September 2020 data. Towns are ranked from highest to lowest 4-room median price; groups with fewer than 30 records are omitted and labels show total counts across the three room types.

**Observation:** Prices vary starkly by location even for the same flat type—4-room medians in central mature towns (such as Central Area, Queenstown and Bukit Merah) exceed 5-room medians in outer non-mature towns (such as Woodlands, Sembawang and Yishun). Within each town, the horizontal span also shows a clear price step-up from 3-room to 4-room and 5-room flats.

In [26]:
# Compare 3-room, 4-room and 5-room median prices across towns for 2016–2020.
room_types = ["3 ROOM", "4 ROOM", "5 ROOM"]
room_colours = {"3 ROOM": "#2ca02c", "4 ROOM": "#9467bd", "5 ROOM": "#d62728"}
town_room_data = df.loc[
    df["transaction_year"].between(2016, 2020) & df["flat_type"].isin(room_types)
]

# Rank towns from highest to lowest 4-room median price and record total counts.
four_room_medians = (
    town_room_data.loc[town_room_data["flat_type"].eq("4 ROOM")]
    .groupby("town")["resale_price"]
    .median()
)
town_room_stats = (
    town_room_data.groupby("town")["resale_price"]
    .agg(["count"])
    .assign(median=four_room_medians)
    .sort_values("median", ascending=False)
)
ordered_towns = list(town_room_stats.index)
town_room_labels = {
    town: f"{town} (n={int(stats['count']):,})"
    for town, stats in town_room_stats.iterrows()
}
y_labels = [town_room_labels[t] for t in ordered_towns]

# Require at least 30 records per town and room type before plotting a median dot.
type_stats = (
    town_room_data.groupby(["town", "flat_type"])["resale_price"]
    .agg(["median", "count"])
    .unstack("flat_type")
)
type_medians = type_stats["median"].where(type_stats["count"] >= 30).reindex(ordered_towns)
type_counts = type_stats["count"].fillna(0).reindex(ordered_towns)

town_room_fig = go.Figure()

# Draw horizontal bars connecting the lowest and highest room-type median in each town.
for town, label in zip(ordered_towns, y_labels):
    row_vals = type_medians.loc[town].dropna()
    if len(row_vals) >= 2:
        town_room_fig.add_trace(go.Scatter(
            x=[row_vals.min(), row_vals.max()],
            y=[label, label],
            mode="lines",
            line=dict(color="#cccccc", width=3),
            showlegend=False,
            hoverinfo="skip",
        ))

# Add coloured median dots for 3-room, 4-room and 5-room flats.
for flat_type in room_types:
    town_room_fig.add_trace(go.Scatter(
        x=type_medians[flat_type],
        y=y_labels,
        mode="markers",
        name=flat_type,
        marker=dict(color=room_colours[flat_type], size=10, line=dict(color="white", width=1)),
        customdata=type_counts[[flat_type]],
        hovertemplate=(
            f"Town: %{{y}}<br>Flat type: {flat_type}<br>"
            "Median: SGD %{x:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>"
        ),
    ))

# Place the town with the highest 4-room median at the top.
town_room_fig.update_yaxes(
    categoryorder="array",
    categoryarray=y_labels,
    autorange="reversed",
    automargin=True,
)
town_room_fig.update_xaxes(tickformat=",.0f")
town_room_fig.update_layout(
    title="Town median prices by room flat type (sorted by 4-room) — Jan 2016–Sep 2020",
    xaxis_title="Median resale price (SGD)",
    yaxis_title="Town (total records across 3-room, 4-room and 5-room)",
    height=900,
    autosize=True,
    template="plotly_white",
    legend_title_text="Room flat type",
)
town_room_fig.show(renderer="plotly_mimetype", config={"responsive": True})

### Price by flat age across sale years

Compare median prices by five-year age band for 4-room, 5-room and executive flats, with one coloured line per sale year for 2016–2020. Bands need at least 30 records; 2020 covers January–September.

**Observation:** Across all three flat types, resale prices do not decline steadily with flat age: 4-room and 5-room prices peak among newer flats, dip for middle-aged flats and rebound at older ages, while executive prices generally rise with age. This shows that the effect of flat age is non-linear and confounded by where and when different flat cohorts were built.

In [27]:
# Use the latest five recorded years and display one panel each for 4-room, 5-room and executive flats.
latest_age_year = int(df["transaction_year"].max())
first_age_year = latest_age_year - 4
age_flat_types = ["4 ROOM", "5 ROOM", "EXECUTIVE"]
age_price_data = df.loc[
    df["transaction_year"].between(first_age_year, latest_age_year)
    & df["flat_type"].isin(age_flat_types)
].dropna(subset=["flat_age_years", "resale_price"])
age_price_data = age_price_data.assign(
    age_band=(age_price_data["flat_age_years"] // 5 * 5).astype(int)
)
age_price_stats = (
    age_price_data.groupby(["flat_type", "transaction_year", "age_band"])["resale_price"]
    .agg(["median", "count"])
)
age_bands = list(
    range(int(age_price_data["age_band"].min()), int(age_price_data["age_band"].max()) + 5, 5)
)
age_labels = [f"{age}–{age + 4}" for age in age_bands]
age_year_colours = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]

age_price_fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{}, {}], [{}, None]],
    subplot_titles=age_flat_types,
    horizontal_spacing=0.10,
    vertical_spacing=0.14,
)

# Add one coloured line per sale year in each flat-type subplot; leave groups below 30 records as gaps.
for panel_index, flat_type in enumerate(age_flat_types):
    row = panel_index // 2 + 1
    col = panel_index % 2 + 1
    type_stats = age_price_stats.loc[flat_type]

    for i, year in enumerate(range(first_age_year, latest_age_year + 1)):
        if year not in type_stats.index.get_level_values("transaction_year"):
            continue
        year_stats = type_stats.loc[year].reindex(age_bands)
        year_label = "2020 (Jan–Sep)" if year == 2020 else str(year)

        age_price_fig.add_trace(
            go.Scatter(
                x=age_labels,
                y=year_stats["median"].where(year_stats["count"] >= 30),
                mode="lines+markers",
                name=year_label,
                legendgroup=year_label,
                showlegend=(panel_index == 0),
                meta=[flat_type, year_label],
                connectgaps=False,
                line={"color": age_year_colours[i]},
                marker={"size": 5},
                customdata=year_stats[["count"]],
                hovertemplate=(
                    "Flat type: %{meta[0]}<br>Year: %{meta[1]}<br>Age: %{x} years<br>"
                    "Median: SGD %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

    age_price_fig.update_xaxes(
        title_text="Approximate flat age (five-year bands)",
        type="category",
        categoryorder="array",
        categoryarray=age_labels,
        tickangle=-45,
        row=row,
        col=col,
    )
    age_price_fig.update_yaxes(
        title_text="Median resale price (SGD)" if col == 1 else "",
        tickformat=",.0f",
        row=row,
        col=col,
    )

age_price_fig.update_layout(
    title=f"Price by flat age — 4-room, 5-room and executive flats, {first_age_year}–{latest_age_year}",
    height=1050,
    autosize=True,
    template="plotly_white",
    legend_title_text="Sale year",
)
age_price_fig.show(renderer="plotly_mimetype", config={"responsive": True})

### Top 10% resale prices versus storey midpoint

Examine the highest-priced 10% of 2016–2020 transactions (≥90th percentile) across storey midpoints, with a median line and middle-50% band for midpoints with at least 30 records.

**Observation:** Among the top 10% highest-priced resale transactions in 2016–2020, prices rise steadily with floor height, especially above the 20th storey where million-dollar resales are concentrated, confirming that high floor levels command a substantial premium in the upper end of the market.

In [28]:
# Filter 2016–2020 transactions at or above the 90th percentile of 2016–2020 resale prices.
recent_storey_data = df.loc[
    df["transaction_year"].between(2016, 2020),
    ["storey_midpoint", "resale_price", "town", "flat_type", "transaction_year"],
].dropna(subset=["storey_midpoint", "resale_price"])

price_p90_threshold = recent_storey_data["resale_price"].quantile(0.90)
storey_p90_data = recent_storey_data.loc[
    recent_storey_data["resale_price"] >= price_p90_threshold
]

# Summarise median and middle-50% prices at each storey midpoint with at least 30 records.
storey_p90_summary = storey_p90_data.groupby("storey_midpoint")["resale_price"].agg(
    median="median",
    q1=lambda prices: prices.quantile(0.25),
    q3=lambda prices: prices.quantile(0.75),
    transactions="count",
)
storey_p90_summary[["median", "q1", "q3"]] = storey_p90_summary[["median", "q1", "q3"]].where(
    storey_p90_summary["transactions"] >= 30, axis=0
)
storey_p90_summary = storey_p90_summary.rename_axis("storey_midpoint").reset_index()

storey_p90_sample = storey_p90_data.sample(
    n=min(10_000, len(storey_p90_data)), random_state=42
)

# Shade the middle 50% of top-10% prices across storey midpoints.
storey_p90_fig = go.Figure(go.Scatter(
    x=storey_p90_summary["storey_midpoint"],
    y=storey_p90_summary["q1"],
    mode="lines",
    line={"width": 0},
    showlegend=False,
    hoverinfo="skip",
    connectgaps=False,
))
storey_p90_fig.add_trace(go.Scatter(
    x=storey_p90_summary["storey_midpoint"],
    y=storey_p90_summary["q3"],
    mode="lines",
    line={"width": 0},
    fill="tonexty",
    fillcolor="rgba(31,119,180,0.2)",
    name="Middle 50% of prices",
    hoverinfo="skip",
    connectgaps=False,
))

# Plot top-10% transactions in the background.
storey_p90_fig.add_trace(go.Scatter(
    x=storey_p90_sample["storey_midpoint"],
    y=storey_p90_sample["resale_price"],
    mode="markers",
    name="Transactions (≥90th percentile, 2016–2020)",
    marker={"size": 4, "color": "#1f77b4", "opacity": 0.25},
    customdata=storey_p90_sample[["town", "flat_type", "transaction_year"]],
    hovertemplate=(
        "Storey midpoint: %{x}<br>Price: SGD %{y:,.0f}<br>"
        "Town: %{customdata[0]}<br>Flat type: %{customdata[1]}<br>"
        "Year: %{customdata[2]}<extra></extra>"
    ),
))

# Overlay the median price line for each storey midpoint.
storey_p90_fig.add_trace(go.Scatter(
    x=storey_p90_summary["storey_midpoint"],
    y=storey_p90_summary["median"],
    mode="lines+markers",
    name="Median price (≥90th percentile)",
    line={"color": "#125183", "width": 3},
    marker={"size": 6},
    connectgaps=False,
    customdata=storey_p90_summary[["transactions", "q1", "q3"]],
    hovertemplate=(
        "Storey midpoint: %{x}<br>Median: SGD %{y:,.0f}<br>"
        "Middle 50%: SGD %{customdata[1]:,.0f}–%{customdata[2]:,.0f}<br>"
        "Transactions: %{customdata[0]:,.0f}<extra></extra>"
    ),
))

storey_p90_fig.update_layout(
    title=f"Top 10% resale prices (≥SGD {price_p90_threshold:,.0f}) vs storey midpoint — Jan 2016–Sep 2020",
    xaxis_title="Storey midpoint",
    yaxis_title="Resale price (SGD)",
    height=650,
    autosize=True,
    template="plotly_white",
    legend={"orientation": "h", "y": 1.08},
)
storey_p90_fig.update_yaxes(tickformat=",.0f")
storey_p90_fig.show(renderer="plotly_mimetype", config={"responsive": True})

## Modelling

Compare models on the same chronological split and select by validation MAE.

| Model | Why chosen | Encoding and transformations required |
|---|---|---|
| **Ridge** | Fast, relatively interpretable reference. Regularisation limits large coefficients from correlated features; comparison with boosting assesses the benefit of more complex patterns. | One-hot encoding; numeric median imputation and standard scaling. |
| **CatBoost** | Suits categorical features such as town and flat type. Captures non-linear effects and interactions through boosted trees and built-in categorical processing. | Text category labels; missing-label replacement; internal categorical encoding and numeric missing-value handling. |
| **LightGBM** | Efficient boosting for this dataset's size. Leaf-wise growth captures detailed patterns, with leaf settings controlling complexity. | Categorical dtypes; unseen labels mapped to missing; internal categorical and missing-value handling. |
| **XGBoost** | Alternative boosting approach that captures non-linear effects and interactions. Depth limits and regularisation control complexity, providing another accuracy comparison. | Categorical dtypes; native categorical support enabled; unseen labels mapped to missing; internal missing-value handling. |


### Configure modelling

Choose features, date range and split ratios. Parameters were tuned using validation MAE. See **Appendix → Hyperparameter tuning** for results.


In [29]:
# Choose the prediction target and recorded date column.
TARGET_COLUMN = "resale_price"
DATE_COLUMN = "month"

# Restrict the modelling period before splitting chronologically.
MODEL_START = "2012-03"
MODEL_END = None  # None uses the latest available month.
SPLIT_RATIOS = (0.70, 0.15, 0.15)

# Numeric inputs are scaled for Ridge, but not for the boosting models.
NUMERIC_FEATURES = [
    "floor_area_sqm",
    "flat_age_years",
    "storey_midpoint",
    "transaction_year",
]

# Treat month-of-year as a category to allow seasonal patterns.
CATEGORICAL_FEATURES = [
    "town",
    "flat_type",
    "flat_model",
    "transaction_month",
    "block_address",
]

# Keep runs reproducible and limit CPU usage.
RANDOM_SEED = 42
CPU_THREADS = min(4, os.cpu_count() or 1)
EARLY_STOPPING_ROUNDS = 50

# Leave the final test disabled until model selection is finished.
RUN_FINAL_TEST = True

# Tuned on validation data from March 2012 onward, including block_address.
MODEL_PARAMETERS = {
    "Ridge": {
        "alpha": 0.1,
        "solver": "lsqr",
        "tol": 1e-05
    },
    "CatBoost": {
        "iterations": 2000,
        "depth": 8,
        "learning_rate": 0.05
    },
    "LightGBM": {
        "n_estimators": 2000,
        "num_leaves": 63,
        "learning_rate": 0.05
    },
    "XGBoost": {
        "n_estimators": 1500,
        "max_depth": 6,
        "learning_rate": 0.05
    }
}

### Check configuration

Check feature lists, target leakage and split ratios.


In [30]:
# Combine features and check for empty or overlapping lists.
feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

assert feature_columns and len(feature_columns) == len(set(feature_columns)), \
    "Choose at least one feature, without duplicates."

# Exclude the target, price-derived features and original date.
assert not set(feature_columns) & {TARGET_COLUMN, "resale_price", "price_per_sqm", DATE_COLUMN}, \
    "Remove price-derived columns, the target and original date from features."

# Require three positive ratios that sum to 1.
assert len(SPLIT_RATIOS) == 3 and min(SPLIT_RATIOS) > 0 and np.isclose(sum(SPLIT_RATIOS), 1), \
    "Provide three positive split ratios that sum to 1."

required_columns = [DATE_COLUMN, TARGET_COLUMN] + feature_columns


### Select the modelling period

Keep transactions from `MODEL_START` to `MODEL_END` (currently March 2012 onward), ready for the chronological split.


In [31]:
# Keep transactions within the configured date range.
period_mask = df[DATE_COLUMN] >= pd.Period(MODEL_START, freq="M")

if MODEL_END is not None:
    period_mask &= df[DATE_COLUMN] <= pd.Period(MODEL_END, freq="M")

modelling_data = df.loc[period_mask, required_columns].copy()

# Remove missing dates/prices and sort before the chronological split.
modelling_data = (
    modelling_data
    .dropna(subset=[DATE_COLUMN, TARGET_COLUMN])
    .sort_values(DATE_COLUMN, kind="stable")
    .reset_index(drop=True)
)

### Split chronologically

Find whole-month boundaries nearest 70/15/15, then show the actual dates and sizes.


In [32]:
# Count records by month and calculate the cumulative share of all records.
monthly_rows = modelling_data.groupby(DATE_COLUMN).size().sort_index()
cumulative_share = monthly_rows.cumsum() / len(modelling_data)

# Find whole-month boundaries closest to 70% and 85% cumulatively.
train_end = (cumulative_share - SPLIT_RATIOS[0]).abs().idxmin()
validation_end = (
    cumulative_share - sum(SPLIT_RATIOS[:2])
).abs().idxmin()

# Earlier months train the model; later months validate and test it.
train_data = modelling_data.loc[
    modelling_data[DATE_COLUMN] <= train_end
]

validation_data = modelling_data.loc[
    (modelling_data[DATE_COLUMN] > train_end)
    & (modelling_data[DATE_COLUMN] <= validation_end)
]

test_data = modelling_data.loc[
    modelling_data[DATE_COLUMN] > validation_end
]

if any(part.empty for part in [train_data, validation_data, test_data]):
    raise ValueError("A set is empty; adjust the date range or split ratios.")

# Report actual percentages because whole-month cuts are approximate.
split_summary = pd.DataFrame([
    {
        "set": name,
        "start": str(part[DATE_COLUMN].min()),
        "end": str(part[DATE_COLUMN].max()),
        "rows": len(part),
        "percent": round(100 * len(part) / len(modelling_data), 2),
    }
    for name, part in [
        ("Training", train_data),
        ("Validation", validation_data),
        ("Test", test_data),
    ]
])

split_summary

,set,start,end,rows,percent
0,Training,2012-03,2018-06,119107,70.37
1,Validation,2018-07,2019-07,24346,14.38
2,Test,2019-08,2020-09,25795,15.24


### Preview split records

Show the first three records from each set after the chronological split.


In [33]:
# Show the first three rows of each set.
for name, data in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Test", test_data),
]:
    print(f"\n{name}")

    with pd.option_context("display.max_columns", None):
        display(data.head(3))


Training


,month,resale_price,floor_area_sqm,flat_age_years,storey_midpoint,transaction_year,town,flat_type,flat_model,transaction_month,block_address
0,2012-03,250000.0,45.0,26,8,2012,ANG MO KIO,2 ROOM,IMPROVED,3,172 ANG MO KIO AVE 4
1,2012-03,265000.0,44.0,32,3,2012,ANG MO KIO,2 ROOM,IMPROVED,3,510 ANG MO KIO AVE 8
2,2012-03,315000.0,68.0,32,8,2012,ANG MO KIO,3 ROOM,NEW GENERATION,3,610 ANG MO KIO AVE 4



Validation


,month,resale_price,floor_area_sqm,flat_age_years,storey_midpoint,transaction_year,town,flat_type,flat_model,transaction_month,block_address
119107,2018-07,202000.0,44.0,40,11,2018,ANG MO KIO,2 ROOM,IMPROVED,7,116 ANG MO KIO AVE 4
119108,2018-07,210000.0,44.0,40,11,2018,ANG MO KIO,2 ROOM,IMPROVED,7,314 ANG MO KIO AVE 3
119109,2018-07,230000.0,68.0,37,8,2018,ANG MO KIO,3 ROOM,NEW GENERATION,7,178 ANG MO KIO AVE 4



Test


,month,resale_price,floor_area_sqm,flat_age_years,storey_midpoint,transaction_year,town,flat_type,flat_model,transaction_month,block_address
143453,2019-08,185000.0,44.0,42,5,2019,ANG MO KIO,2 ROOM,IMPROVED,8,323 ANG MO KIO AVE 3
143454,2019-08,190000.0,45.0,33,2,2019,ANG MO KIO,2 ROOM,IMPROVED,8,174 ANG MO KIO AVE 4
143455,2019-08,200000.0,44.0,39,8,2019,ANG MO KIO,2 ROOM,IMPROVED,8,508 ANG MO KIO AVE 8


### Prepare features and targets

Apply the same numeric and text conversions to each set without learning from validation or test data.


In [34]:
def prepare_inputs(data):
    # Select only the configured predictors, keeping the target separate.
    features = data[feature_columns].copy()

    # Convert numeric columns together; treat infinities as missing.
    features[NUMERIC_FEATURES] = (
        features[NUMERIC_FEATURES]
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )

    # CatBoost and the Ridge encoder receive consistent text labels.
    for column in CATEGORICAL_FEATURES:
        features[column] = (
            features[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(object)
        )

    return features, data[TARGET_COLUMN].astype(float)


# Apply identical conversions without learning from later sets.
X_train, y_train = prepare_inputs(train_data)
X_validation, y_validation = prepare_inputs(validation_data)
X_test, y_test = prepare_inputs(test_data)

### Prepare native categories

For LightGBM and XGBoost, use training-only category levels; unseen labels become missing. Ridge and CatBoost use the original text inputs.


In [35]:
# Learn the available category labels from training data only.
native_category_types = {
    column: pd.CategoricalDtype(
        categories=sorted(X_train[column].unique())
    )
    for column in CATEGORICAL_FEATURES
}


def native_inputs(features):
    # Build the separate categorical representation for LightGBM and XGBoost.
    features = features.copy()

    for column, dtype in native_category_types.items():
        # Unseen labels become missing instead of introducing new categories.
        known = features[column].isin(dtype.categories)
        features[column] = features[column].where(known).astype(dtype)

    return features


X_train_native = native_inputs(X_train)
X_validation_native = native_inputs(X_validation)
X_test_native = native_inputs(X_test)

### Plot prediction accuracy

Plot actual versus predicted validation prices. The dashed line represents perfect predictions.


In [36]:
def plot_predictions(name, actual, predicted):
    # Use the same price limits on both axes.
    limits = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]

    fig = go.Figure(go.Scattergl(
        x=actual,
        y=predicted,
        mode="markers",
        marker=dict(color="#1f77b4", size=4, opacity=0.3),
        hovertemplate="Actual: SGD %{x:,.0f}, predicted: SGD %{y:,.0f}<extra></extra>",
    ))

    # Prices on this line are predicted exactly.
    fig.add_trace(go.Scatter(
        x=limits,
        y=limits,
        mode="lines",
        line=dict(color="grey", dash="dash"),
        hoverinfo="skip",
    ))

    fig.update_layout(
        title=f"{name}: actual vs predicted prices (validation)",
        xaxis=dict(title="Actual resale price (SGD)", range=limits),
        yaxis=dict(title="Predicted resale price (SGD)", range=limits),
        template="plotly_white",
        height=360,
        autosize=True,
        showlegend=False,
        margin=dict(l=65, r=25, t=55, b=55),
    )
    fig.show(config={"responsive": True})


### Shared training and evaluation

Fit each model, store validation scores and show its prediction scatter plot. No test labels are used.


In [37]:
def regression_scores(name, actual, predicted, seconds):
    # Keep the same metrics for every model so comparisons are consistent.
    return {
        "Model": name,
        "MAE_SGD": mean_absolute_error(actual, predicted),
        "RMSE_SGD": root_mean_squared_error(actual, predicted),
        "R2": r2_score(actual, predicted),
        "Fit_seconds": seconds,
    }


# Store each model and its scores once, keyed by model name.
fitted_models = {}
validation_scores = {}


def train_and_validate(
    name,
    model,
    training_features,
    validation_features,
    **fit_options,
):
    # Fit using training labels and any validation early-stopping options.
    start = perf_counter()
    model.fit(training_features, y_train, **fit_options)
    elapsed = perf_counter() - start

    # Calculate validation metrics once for display and later comparison.
    predicted = model.predict(validation_features)
    scores = regression_scores(name, y_validation, predicted, elapsed)

    fitted_models[name] = model
    validation_scores[name] = scores

    plot_predictions(name, y_validation, predicted)

    return pd.DataFrame([scores])


### Ridge regression

Fit numeric imputation/scaling and sparse one-hot encoding inside the training-only pipeline. Show validation predictions against actual prices.


In [38]:
# Learn numeric replacements and scaling from training data inside the pipeline.
numeric_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("scaler", StandardScaler()),
])

# One-hot encode categories; unseen labels produce zero indicator values.
ridge_preprocessing = ColumnTransformer([
    ("numeric", numeric_preprocessing, NUMERIC_FEATURES),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=True),
        CATEGORICAL_FEATURES,
    ),
], sparse_threshold=1.0)

# Fit preprocessing and regression together to avoid data leakage.
ridge_model = Pipeline([
    ("preprocessing", ridge_preprocessing),
    ("regressor", Ridge(**MODEL_PARAMETERS["Ridge"])),
])

train_and_validate("Ridge", ridge_model, X_train, X_validation)


,Model,MAE_SGD,RMSE_SGD,R2,Fit_seconds
0,Ridge,29199.088124,41024.752528,0.930311,0.667297


### CatBoost regression

Use text categories directly on CPU; validation MAE controls early stopping and the best iteration. Show validation predictions against actual prices.


In [39]:
# CatBoost handles the configured text categories without external encoding.
catboost_model = CatBoostRegressor(
    **MODEL_PARAMETERS["CatBoost"],
    loss_function="RMSE",
    eval_metric="MAE",
    cat_features=CATEGORICAL_FEATURES,
    task_type="CPU",
    thread_count=CPU_THREADS,
    random_seed=RANDOM_SEED,
    allow_writing_files=False,
    verbose=False,
)

# Validation MAE controls stopping; retain the best iteration.
train_and_validate(
    "CatBoost",
    catboost_model,
    X_train,
    X_validation,
    eval_set=(X_validation, y_validation),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    use_best_model=True,
)


,Model,MAE_SGD,RMSE_SGD,R2,Fit_seconds
0,CatBoost,25974.416112,36056.66816,0.946168,62.366033


### LightGBM regression

Use native categories on CPU with validation-MAE early stopping; no scaling is needed. Show validation predictions against actual prices.


In [40]:
# LightGBM receives the training-defined categorical representation.
lightgbm_model = LGBMRegressor(
    **MODEL_PARAMETERS["LightGBM"],
    objective="regression",
    metric="l1",
    device_type="cpu",
    n_jobs=CPU_THREADS,
    random_state=RANDOM_SEED,
    deterministic=True,
    force_col_wise=True,
    verbosity=-1,
)

# Stop when validation MAE no longer improves; suppress iteration logs.
train_and_validate(
    "LightGBM",
    lightgbm_model,
    X_train_native,
    X_validation_native,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_validation_native, y_validation)],
    callbacks=[
        early_stopping(
            EARLY_STOPPING_ROUNDS,
            first_metric_only=True,
            verbose=False,
        ),
        log_evaluation(0),
    ],
)


/Users/gin/Documents/Job/fulltime/htx/.venv_test/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,Model,MAE_SGD,RMSE_SGD,R2,Fit_seconds
0,LightGBM,24723.445566,34806.315049,0.949837,13.004905


### XGBoost regression

Use CPU histogram training and native categories; validation MAE controls early stopping. Show validation predictions against actual prices.


In [41]:
# Use CPU histogram training with native categorical support.
xgboost_model = XGBRegressor(
    **MODEL_PARAMETERS["XGBoost"],
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    device="cpu",
    enable_categorical=True,
    n_jobs=CPU_THREADS,
    random_state=RANDOM_SEED,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

# Validation, not the final test set, controls early stopping.
train_and_validate(
    "XGBoost",
    xgboost_model,
    X_train_native,
    X_validation_native,
    eval_set=[(X_validation_native, y_validation)],
    verbose=False,
)


,Model,MAE_SGD,RMSE_SGD,R2,Fit_seconds
0,XGBoost,24889.926381,35651.834223,0.94737,21.096675


### Compare validation results

Rank models by validation MAE; lower MAE/RMSE and higher R² are better.


In [42]:
# Collect all validation results and rank by the primary metric, MAE.
validation_results = (
    pd.DataFrame(validation_scores.values())
    .sort_values("MAE_SGD")
    .reset_index(drop=True)
)

# Select using validation only; rounding affects display, not ranking.
selected_model_name = validation_results.iloc[0]["Model"]
print(f"Lowest validation MAE: {selected_model_name}")

validation_results.round({
    "MAE_SGD": 2,
    "RMSE_SGD": 2,
    "R2": 4,
    "Fit_seconds": 2,
})


Lowest validation MAE: LightGBM


,Model,MAE_SGD,RMSE_SGD,R2,Fit_seconds
0,LightGBM,24723.45,34806.32,0.9498,13.00
1,XGBoost,24889.93,35651.83,0.9474,21.10
2,CatBoost,25974.42,36056.67,0.9462,62.37
3,Ridge,29199.09,41024.75,0.9303,0.67


### Final test evaluation

Evaluate all models on the test set. Set `RUN_FINAL_TEST=True` in **Configure modelling**, then rerun from that cell downward to use the latest settings. Model selection remains based on validation MAE.


In [43]:
if RUN_FINAL_TEST:
    test_scores = []
    training_times = validation_results.set_index("Model")["Fit_seconds"]

    # Evaluate every fitted model using its expected input format.
    for name, model in fitted_models.items():
        features = X_test_native if name in {"LightGBM", "XGBoost"} else X_test
        predicted = model.predict(features)

        test_scores.append(
            regression_scores(name, y_test, predicted, training_times[name])
        )

    # Compare test scores without changing the validation-selected model.
    test_results = pd.DataFrame(test_scores).sort_values("MAE_SGD").reset_index(drop=True)

    display(test_results.round({
        "MAE_SGD": 2,
        "RMSE_SGD": 2,
        "R2": 4,
        "Fit_seconds": 2,
    }))

else:
    print("Test evaluation disabled. Set RUN_FINAL_TEST = True to evaluate all models.")


,Model,MAE_SGD,RMSE_SGD,R2,Fit_seconds
0,CatBoost,28986.85,39922.48,0.9311,62.37
1,LightGBM,29616.73,42455.71,0.9221,13.00
2,XGBoost,32555.23,49015.53,0.8961,21.10
3,Ridge,35565.30,49162.52,0.8955,0.67


### Model selection: LightGBM

**LightGBM is selected** for its lowest validation MAE (SGD 24,723.45), competitive test MAE (SGD 29,616.73) and efficient training. CatBoost achieved a test MAE about SGD 630 lower, but took approximately five times longer to train. Selection remains based on validation MAE.


## Feature importance

Examine which inputs the selected LightGBM model relies on for its predictions.


### LightGBM permutation importance

Shuffle each feature five times on validation data and measure the increase in MAE. Larger increases indicate greater predictive importance. Correlated features such as town and block address share information; importance does not establish causation.


In [44]:
# Measure importance using the fitted LightGBM model and all validation rows.
result = permutation_importance(
    fitted_models["LightGBM"],
    X_validation_native,
    y_validation,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_SEED,
    n_jobs=1,
)

# Retain full precision for plotting; round only the displayed table.
feature_importance = pd.DataFrame({
    "feature": X_validation_native.columns,
    "mae_increase": result.importances_mean,
    "std": result.importances_std,
}).sort_values("mae_increase", ascending=False).reset_index(drop=True)

feature_importance.round(2)


,feature,mae_increase,std
0,town,72346.49,387.07
1,floor_area_sqm,63003.76,293.28
2,flat_age_years,20518.48,72.74
3,flat_type,17088.74,114.90
4,flat_model,11431.83,98.72
5,block_address,11419.48,83.29
6,storey_midpoint,9920.07,97.19
7,transaction_month,16.25,17.53
8,transaction_year,0.00,0.00


### Plot LightGBM feature importance

Show the increase in validation MAE when each feature is shuffled. Error bars show standard deviation across five shuffles, not confidence intervals.

**Observation:** Town, floor area and flat age produced the largest increases in validation MAE when shuffled. Transaction month and transaction year have minimal importance in this model, showing that HDB prices do not vary strongly with seasonal or year-to-year changes over the 2012–2020 period compared with physical and location attributes.


In [45]:
# Place the most important features at the top.
importance_plot = feature_importance.sort_values("mae_increase")

fig = go.Figure(go.Bar(
    x=importance_plot["mae_increase"],
    y=importance_plot["feature"],
    orientation="h",
    marker_color="#1f77b4",
    error_x=dict(type="data", array=importance_plot["std"], color="#555555"),
    hovertemplate="%{y}: SGD %{x:,.0f} MAE increase<extra></extra>",
))

fig.update_layout(
    title="LightGBM permutation importance",
    xaxis_title="Increase in validation MAE (SGD)",
    xaxis=dict(zeroline=True, zerolinecolor="grey"),
    template="plotly_white",
    height=420,
    autosize=True,
    showlegend=False,
    margin=dict(l=155, r=30, t=60, b=60),
)
fig.show(config={"responsive": True})


## Key factors and strategies to curb housing price inflation

#### Key factors affecting HDB resale prices

1. **Location (`town` and `block_address`):** Location is the primary determinant of resale value. Town ranks as the most influential predictor in the permutation importance analysis, while incorporating block-level address information reduced validation error by 9.8% to 27.1% across all evaluated models. As observed in the town price comparison, median 4-room prices in central towns (Central Area, Queenstown and Bukit Merah) surpass median 5-room prices in outer towns (Woodlands, Sembawang and Yishun), indicating substantial price stratification at both the town and individual block level.
2. **Flat size (`floor_area_sqm` and `flat_type`):** Floor area is the second strongest predictor of resale price ($r = 0.62$) and flat type ranks fourth. While total transaction price increases with floor area and room category, unit price trends since 2008 show that 2-room and 3-room flats command higher median prices per square metre than larger flat types.
3. **Flat age and lease duration (`flat_age_years`):** Flat age ranks third in predictive importance and exhibits a non-linear relationship with price. For 4-room and 5-room flats, median prices peak among newer flats aged 5–14 years, decline across middle-aged cohorts (15–34 years) and rebound among older flats aged 35–44 years, whereas executive flat prices generally increase with age.
4. **Flat design and floor level (`flat_model` and `storey_midpoint`):** Flat model and storey height contribute moderately to price variation, with higher storeys positively correlated with resale price. By contrast, transaction month and transaction year show negligible importance over the 2012–2020 modelling window, confirming that physical and locational attributes outweigh short-term seasonal or annual shifts during this period.

#### Recommended strategies to curb housing price inflation

- **Monitor resale price premiums in high-demand towns (`town`, `block_address`, `flat_age_years`):** Regularly assess whether existing housing policies, such as longer MOPs and subsidy recovery, effectively limit excessive resale price increases.
- **Improve housing supply and amenities in less central towns (`town`):** Build more flats and improve public transport and nearby amenities in lower-priced towns to make them more attractive to buyers. This can reduce demand pressure in central locations and help narrow price differences between towns.
- **Reduce the influence of unusually high resale prices (`block_address`, `flat_model`, `storey_midpoint`):** Publish reference prices that account for differences in floor level, flat type, and age, so buyers and sellers can make fairer price comparisons. This helps prevent expensive transactions, such as high-floor or premium flats, from driving up price expectations for other flats in the same block or neighbourhood.

## Model Prediction Task II: In-house predictive model considerations

Factors and considerations when deploying an in-house HDB resale price model for users:

| Factor | Key considerations |
|---|---|
| **Users and explainability** | Tailor outputs to different users: buyers and sellers need estimated resale prices and key price drivers, while policy officers require trend analysis and feature-level insights. Present predicted prices alongside calibrated uncertainty ranges rather than relying solely on point estimates. |
| **Data freshness and retraining** | Regularly incorporate newly registered HDB resale transactions and retrain using recent data to reflect changing market conditions. Historical testing showed that including older transactions increased test MAE to SGD 57,810–104,119, highlighting the importance of data recency. Ensure all features are available at prediction time to prevent data leakage. |
| **Unseen and newly MOP-ed blocks** | Ensure the model handles newly eligible resale blocks and previously unseen addresses. Implement fallback handling using features such as `town`, `flat_model`, `floor_area_sqm`, and `flat_age_years` to maintain prediction reliability when block-specific information is unavailable. |
| **Feature enrichment** | Incorporate location-based features such as proximity to MRT stations, schools, and amenities, alongside market indicators such as the HDB Resale Price Index and interest rates. These may improve predictions beyond existing categorical location features. |
| **Model performance and efficiency** | Balance predictive accuracy, training costs, and response times. LightGBM is suitable for CPU-based deployment, with approximately 13-second training times and millisecond-level inference, supporting frequent retraining and responsive user queries. |
| **Monitoring and governance** | Regularly evaluate prediction accuracy using MAE and RMSE across different towns, flat types, and time periods. Identify areas where the model consistently overestimates or underestimates prices, and update the model when performance declines or market conditions change. |

## Appendix

### Feature selection

#### Add `block_address` as a categorical feature (March 2012 onward)

March 2012–September 2020; chronological 70/15/15 split and baseline settings (600 rounds, learning rate 0.05; Ridge `alpha=10`). Only `block_address` was added; test set not evaluated.

| Model | MAE without (SGD) | MAE with (SGD) | MAE reduction | RMSE with (SGD) | R² with |
|---|---|---|---|---|---|
| **XGBoost** | **29,885.81** | **25,867.79** | **13.44%** | **36,598.50** | **0.9445** |
| LightGBM | 29,962.47 | 25,937.39 | 13.43% | 36,509.47 | 0.9448 |
| CatBoost | 32,880.16 | 29,644.58 | 9.84% | 41,137.47 | 0.9299 |
| Ridge | 45,378.24 | 33,081.29 | 27.10% | 46,284.43 | 0.9113 |

**Decision:** Include `block_address`. It reduced validation MAE by 9.8%–27.1% across all models by capturing block-level location differences within each town.

### Chronological split comparison

Compare earlier start dates (`1990-01` and `2000-01`) against the `2012-03` reference using the same features (with `block_address`), baseline parameters and 70/15/15 whole-month split.

#### Chronology test 1: January 1990 onward (`MODEL_START = "1990-01"`)

| Set | From | To | Rows | Share |
|---|---|---|---|---|
| Training | 1990-01 | 2009-06 | 578,041 | 70.09% |
| Validation | 2009-07 | 2014-07 | 122,414 | 14.84% |
| Test | 2014-08 | 2020-09 | 124,305 | 15.07% |

| Set | Model | MAE (SGD) | RMSE (SGD) | R² | Fit time (s) |
|---|---|---|---|---|---|
| **Validation** | **LightGBM** | **88,759.05** | **105,867.30** | **0.2063** | **21.43** |
| Validation | CatBoost | 90,864.93 | 108,633.25 | 0.1643 | 84.90 |
| Validation | XGBoost | 91,630.26 | 109,194.65 | 0.1557 | 17.18 |
| Validation | Ridge | 105,079.97 | 120,348.25 | -0.0256 | 2.24 |
| **Test** | **Ridge** | **82,738.10** | **113,128.34** | **0.4068** | **2.24** |
| Test | CatBoost | 102,567.25 | 129,593.04 | 0.2215 | 84.90 |
| Test | LightGBM | 104,119.45 | 126,999.03 | 0.2524 | 21.43 |
| Test | XGBoost | 107,971.54 | 133,164.07 | 0.1780 | 17.18 |

#### Chronology test 2: January 2000 onward (`MODEL_START = "2000-01"`)

| Set | From | To | Rows | Share |
|---|---|---|---|---|
| Training | 2000-01 | 2012-05 | 375,970 | 69.83% |
| Validation | 2012-06 | 2016-12 | 82,252 | 15.28% |
| Test | 2017-01 | 2020-09 | 80,164 | 14.89% |

| Set | Model | MAE (SGD) | RMSE (SGD) | R² | Fit time (s) |
|---|---|---|---|---|---|
| **Validation** | **LightGBM** | **33,454.81** | **44,586.65** | **0.8824** | **18.09** |
| Validation | XGBoost | 35,134.21 | 48,945.13 | 0.8583 | 15.06 |
| Validation | CatBoost | 38,081.36 | 50,789.17 | 0.8474 | 59.28 |
| Validation | Ridge | 57,477.09 | 74,026.92 | 0.6759 | 1.30 |
| **Test** | **LightGBM** | **57,810.31** | **70,591.76** | **0.7886** | **18.09** |
| Test | XGBoost | 58,464.36 | 74,834.95 | 0.7625 | 15.06 |
| Test | CatBoost | 64,546.01 | 77,998.90 | 0.7420 | 59.28 |
| Test | Ridge | 82,873.03 | 94,880.66 | 0.6182 | 1.30 |

**Decision:** Keep March 2012 onward. Using data from 1990 or 2000 causes much higher prediction errors because models trained on older price levels do not reflect recent market prices, and records before March 2012 use approval dates instead of registration dates.

### Hyperparameter tuning

March 2012–September 2020 with categorical `block_address` (training to June 2018; validation July 2018–July 2019). 31 sampled configurations (4 Ridge + 9 per boosting model), seed 42, 50-round early stopping. Ranked by validation MAE; test data not used.

| Model | Baseline parameters | Values tested |
|---|---|---|
| **Ridge** | `alpha=10`, `solver="lsqr"`, `tol=1e-5` | `alpha`: 0.1, 1, 10, 100 |
| **CatBoost** | `iterations=600`, `depth=6`, `learning_rate=0.05` | `iterations`: 600, 1,500, 2,000; `depth`: 4, 6, 8; `learning_rate`: 0.03, 0.05, 0.1 |
| **LightGBM** | `n_estimators=600`, `num_leaves=31`, `learning_rate=0.05` | `n_estimators`: 600, 1,500, 2,000; `num_leaves`: 15, 31, 63; `learning_rate`: 0.03, 0.05, 0.1 |
| **XGBoost** | `n_estimators=600`, `max_depth=6`, `learning_rate=0.05` | `n_estimators`: 600, 1,500, 2,000; `max_depth`: 4, 6, 8; `learning_rate`: 0.03, 0.05, 0.1 |

#### Ridge

| alpha | MAE (SGD) | RMSE (SGD) | R² | Fit time (s) |
| --- | --- | --- | --- | --- |
| **0.1** | **29,199.09** | **41,024.75** | **0.9303** | **0.95** |
| 1 | 29,380.08 | 41,287.52 | 0.9294 | 0.51 |
| 10 | 33,081.29 | 46,284.43 | 0.9113 | 0.43 |
| 100 | 42,443.88 | 58,348.66 | 0.8590 | 0.28 |


#### CatBoost

| iterations | depth | learning_rate | Best round | MAE (SGD) | RMSE (SGD) | R² | Fit time (s) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 600 | 4 | 0.03 | 599 | 34,582.06 | 47,542.13 | 0.9064 | 11.55 |
| 600 | 6 | 0.05 | 598 | 29,644.58 | 41,137.47 | 0.9299 | 18.82 |
| 600 | 8 | 0.1 | 599 | 26,948.77 | 37,397.57 | 0.9421 | 24.90 |
| 1500 | 8 | 0.03 | 1,500 | 27,259.49 | 37,844.22 | 0.9407 | 63.31 |
| 1500 | 6 | 0.05 | 1,500 | 27,628.25 | 38,403.36 | 0.9389 | 43.30 |
| 1500 | 4 | 0.1 | 1,499 | 28,523.93 | 39,818.29 | 0.9343 | 28.99 |
| 2000 | 4 | 0.03 | 2,000 | 30,266.87 | 42,114.30 | 0.9266 | 38.32 |
| **2000** | **8** | **0.05** | **2,000** | **25,974.42** | **36,056.67** | **0.9462** | **85.91** |
| 2000 | 6 | 0.1 | 1,979 | 26,306.40 | 36,520.43 | 0.9448 | 62.42 |

#### LightGBM

| n_estimators | num_leaves | learning_rate | Best round | MAE (SGD) | RMSE (SGD) | R² | Fit time (s) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 600 | 15 | 0.03 | 600 | 29,035.83 | 40,716.39 | 0.9314 | 6.19 |
| 600 | 31 | 0.05 | 600 | 25,937.39 | 36,509.47 | 0.9448 | 8.23 |
| 600 | 63 | 0.1 | 600 | 24,989.97 | 35,204.29 | 0.9487 | 7.82 |
| 1500 | 63 | 0.03 | 1,500 | 24,877.05 | 35,028.00 | 0.9492 | 21.03 |
| 1500 | 31 | 0.05 | 1,500 | 25,124.12 | 35,398.68 | 0.9481 | 14.99 |
| 1500 | 15 | 0.1 | 1,500 | 25,370.76 | 35,845.46 | 0.9468 | 10.19 |
| 2000 | 15 | 0.03 | 2,000 | 25,888.10 | 36,669.96 | 0.9443 | 18.66 |
| **2000** | **63** | **0.05** | **1,506** | **24,723.45** | **34,806.32** | **0.9498** | **18.85** |
| 2000 | 31 | 0.1 | 1,476 | 24,943.06 | 35,147.35 | 0.9488 | 11.54 |

#### XGBoost

| n_estimators | max_depth | learning_rate | Best round | MAE (SGD) | RMSE (SGD) | R² | Fit time (s) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 600 | 4 | 0.03 | 600 | 31,145.51 | 41,348.25 | 0.9292 | 6.36 |
| 600 | 6 | 0.05 | 600 | 25,867.79 | 36,598.50 | 0.9445 | 14.41 |
| 600 | 8 | 0.1 | 481 | 25,141.47 | 36,351.71 | 0.9453 | 23.50 |
| 1500 | 8 | 0.03 | 1,498 | 25,123.29 | 36,389.63 | 0.9452 | 66.73 |
| **1500** | **6** | **0.05** | **1,500** | **24,889.93** | **35,651.83** | **0.9474** | **33.44** |
| 1500 | 4 | 0.1 | 1,500 | 24,981.53 | 35,562.48 | 0.9476 | 15.99 |
| 2000 | 4 | 0.03 | 2,000 | 26,311.17 | 36,798.46 | 0.9439 | 21.48 |
| 2000 | 8 | 0.05 | 1,020 | 25,095.66 | 36,319.55 | 0.9454 | 42.94 |
| 2000 | 6 | 0.1 | 821 | 24,989.37 | 35,801.15 | 0.9469 | 17.38 |

#### Selected configurations

| Model | Parameters | Validation MAE (SGD) |
|---|---|---|
| **LightGBM** | `n_estimators=2000`, `num_leaves=63`, `learning_rate=0.05` | **24,723.45** |
| XGBoost | `n_estimators=1500`, `max_depth=6`, `learning_rate=0.05` | 24,889.93 |
| CatBoost | `iterations=2000`, `depth=8`, `learning_rate=0.05` | 25,974.42 |
| Ridge | `alpha=0.1` | 29,199.09 |

**Decision:** Apply each model's best configuration above and select **LightGBM** as the final model (lowest validation MAE at SGD 24,723.45 and fastest boosting fit time).